In [15]:
import pandas as pd

dados = pd.read_csv('tab_Cantareira_2000-01-01_ate_2025-09-16.csv',sep=';', parse_dates=['Data'], dayfirst=True)
dados = pd.concat([dados.filter(like='Cachoeira'),dados['Data']], axis=1)
dados.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9391 entries, 0 to 9390
Data columns (total 13 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   Cachoeira_Nivel                 9391 non-null   object        
 1   Cachoeira_Volume                9391 non-null   object        
 2   Cachoeira_Chuva                 9391 non-null   object        
 3   Cachoeira_ChuvaAcumuladaMensal  9391 non-null   object        
 4   Cachoeira_QJusante              9391 non-null   object        
 5   Cachoeira_VolumeMaximo          9391 non-null   object        
 6   Cachoeira_VolumeMinimo          9391 non-null   object        
 7   Cachoeira_VolumePorcentagem     9391 non-null   object        
 8   Cachoeira_VolumeOperacional     9391 non-null   object        
 9   Cachoeira_VolumeTotal           9391 non-null   object        
 10  Cachoeira_VazaoNatural          9391 non-null   object        
 11  Cach

In [ ]:
# ==============================================================================
# 1. IMPORTS E CONFIGURAÇÕES
# ==============================================================================
import os
import json
import time
import zlib  # Essencial para descompactar a resposta da API
import requests
import pandas as pd
from datetime import date, datetime, timedelta
import calendar

# A nova API usa um certificado SSL que pode falhar na verificação.
# O código abaixo desativa os avisos de segurança ao usar `verify=False`.
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

print("Bibliotecas importadas e configuração concluída.")
year_final = datetime.today().year
month_final = datetime.today().month
day_final = datetime.today().day

dados = pd.read_csv('./tab_cantareira.csv', sep=';', parse_dates=['Data'], dayfirst=True)

ultima_data = dados['Data'].max() + timedelta(days=1)
year = ultima_data.year
month = ultima_data.month
day = ultima_data.day

# ==============================================================================
# 2. PARÂMETROS DA CONSULTA
# ==============================================================================
#DATA_INICIO = date(2000, 1, 1)
DATA_INICIO = date(year, month, day)
DATA_FIM = date(year_final, month_final, day_final)
SISTEMA_ID = 0  # 0 = Sistema Cantareira

# ==============================================================================
# 3. FUNÇÕES DA NOVA DOCUMENTAÇÃO (Refatoradas para maior clareza)
# ==============================================================================

def get_json(url):
    """
    Busca os dados na API e retorna o JSON.
    A API agora envia JSON puro, então a descompressão com zlib não é mais necessária.
    """
    try:
        print(f"  Fazendo requisição para: {url}")
        r = requests.get(url, verify=False, timeout=60)
        r.raise_for_status()  # Verifica se houve erro na requisição (4xx ou 5xx)
        
        # A MUDANÇA ESTÁ AQUI: Usamos o método .json() direto da resposta
        data = r.json() 
        
        return json.dumps(data) # json.dumps ainda é útil para manter a estrutura
    except requests.exceptions.RequestException as e:
        print(f"  !!! Erro de requisição: {e}")
        return None
    except json.JSONDecodeError as e:
        print(f"  !!! Erro ao decodificar JSON. A resposta não parece ser um JSON válido: {e}")
        print(f"      Conteúdo recebido: {r.text[:200]}") # Mostra o início da resposta
        return None

def json2df(jsn):
    df = pd.read_json(jsn)
    return df.drop(['FlagHasError', 'Message'], axis=1, errors='ignore')

def rename_field(x):
    return str(x).replace('/', '-').replace(' (', '-').replace('-', '_').replace(')', '').replace('Cesp', 'CESP').replace('Represa ', '').replace(' ', '')

def list_represas(df):
    lst = df.loc['ListaRepresas']['ReturnObj']
    df_represas = pd.json_normalize(lst)
    return df_represas.drop(['temChuva','temNivel', 'temQjus', 'temQnat', 'temVolume'], axis=1, errors='ignore')

def safe_merge(df_full, df_new):
    """Função auxiliar para fazer o merge de forma segura, tratando o primeiro caso."""
    if df_full.empty:
        return df_new
    else:
        # Usar how='outer' para não perder dados se as datas não alinharem perfeitamente
        return pd.merge(df_full, df_new, on='Data', how='outer')

def list_volumes(df):
    lst = df.loc['ListaDados']['ReturnObj']
    
    # --- NOVA CORREÇÃO APLICADA AQUI ---
    # Iteramos manualmente para criar uma lista "limpa" de registros,
    # ignorando qualquer valor 'None' que apareça dentro da lista 'Dados'.
    records_limpos = []
    for item_diario in lst:
        # Pula dias que não têm a lista 'Dados'
        if not isinstance(item_diario.get('Dados'), list):
            continue
        # Pega apenas os dicionários válidos de dentro da lista 'Dados'
        for record in item_diario['Dados']:
            if isinstance(record, dict):
                records_limpos.append(record)

    # Se não houver nenhum registro válido, retorna um DataFrame vazio
    if not records_limpos:
        return pd.DataFrame()
    
    # Cria o DataFrame a partir da lista de registros já limpa e plana
    df_normalized = pd.DataFrame(records_limpos)
    # --- FIM DA CORREÇÃO ---

    fields = sorted(list(set(df_normalized['Nome'])))
    df_full = pd.DataFrame()

    for field_name in fields:
        j = rename_field(field_name)
        temp_df = df_normalized[df_normalized['Nome'] == field_name].copy()
        temp_df.drop(['FlagConsolidado', 'NAMaxMax', 'NAMinMin', 'QJusanteMax', 'QJusanteMin', 'NivelUltimoDia', 'SistemaId', 'ComponenteId', 'UltimoDia', 'VazaoJusantePrincipal', 'VazaoJusanteSecundaria', 'VolumeOperacionalUltimoDia', 'VolumePorcentagemUltimoDia', 'VolumeTotalUltimoDia', 'Nome'], axis=1, errors='ignore', inplace=True)
        temp_df.columns = [col if col == 'Data' else f"{j}_{col}" for col in temp_df.columns]
        temp_df['Data'] = pd.to_datetime(temp_df['Data'])
        df_full = safe_merge(df_full, temp_df)
    
    if not df_full.empty:
        df_full.set_index('Data', inplace=True)
    return df_full

def list_vazao(df):
    df_represas = list_represas(df)
    lst = df.loc['ListaDados']['ReturnObj']

    # --- NOVA CORREÇÃO APLICADA AQUI ---
    # Lógica idêntica à de list_volumes, mas para a chave 'Qnat'
    records_limpos = []
    for item_diario in lst:
        if not isinstance(item_diario.get('Qnat'), list):
            continue
        for record in item_diario['Qnat']:
            if isinstance(record, dict):
                records_limpos.append(record)

    if not records_limpos:
        return pd.DataFrame()
        
    df_normalized = pd.DataFrame(records_limpos)
    # --- FIM DA CORREÇÃO ---
    
    df_merged = pd.merge(df_normalized, df_represas, on='ComponenteId', how='outer')
    fields = sorted(list(set(df_merged['Nome'].dropna())))
    df_full = pd.DataFrame()

    for field_name in fields:
        j = rename_field(field_name)
        temp_df = df_merged[df_merged['Nome'] == field_name].copy()
        temp_df.drop(['ComponenteId', 'Nome', 'VazaoAfluenteMax', 'VazaoAfluenteMin', 'VazaoNaturalMax', 'VazaoNaturalMin'], axis=1, errors='ignore', inplace=True)
        temp_df.columns = [col if col == 'Data' else f"{j}_{col}" for col in temp_df.columns]
        temp_df['Data'] = pd.to_datetime(temp_df['Data'])
        df_full = safe_merge(df_full, temp_df)
        
    if not df_full.empty:
        df_full.set_index('Data', inplace=True)
    return df_full

def list_SE(df):
    lst = df.loc['ListaDados']['ReturnObj']
    df_se = pd.json_normalize(lst)
    df_se.drop(['Dados', 'Data', 'Qnat'], axis=1, errors='ignore', inplace=True)
    col = [f"SE_{c}" for c in df_se.columns]
    col = [c.replace('SistemaEquivalente.', '').replace('SE_Data', 'Data') for c in col]
    df_se.columns = col
    df_se['Data'] = pd.to_datetime(df_se['Data'])
    df_se.set_index('Data', inplace=True)
    return df_se

def list_SC(df):
    lst = df.loc['ListaDadosSistema']['ReturnObj']
    df_sc = pd.json_normalize(lst)
    df_sc.drop(['objSistema.SistemaId', 'objQETA', 'objSistema.Data'], axis=1, errors='ignore', inplace=True)
    col = [f"SC_{c}".replace('objSistema', '').replace('.', '') for c in df_sc.columns]
    col = [c.replace('SC_Data', 'Data') for c in col]
    df_sc.columns = col
    df_sc['Data'] = pd.to_datetime(df_sc['Data'])
    df_sc.set_index('Data', inplace=True)
    return df_sc

def list_vazaoestruturas(df):
    lst = df.loc['ListaDadosLocais']['ReturnObj']
    list_d = [parte2 for parte1 in lst for parte2 in parte1['Dados'] if isinstance(parte2, dict)]
    if not list_d: return pd.DataFrame()
    df_normalized = pd.DataFrame(list_d)
    fields = sorted(list(set(df_normalized['Abreviatura'])))
    df_full = pd.DataFrame()

    for field_name in fields:
        j = rename_field(field_name)
        temp_df = df_normalized[df_normalized['Abreviatura'] == field_name].copy()
        temp_df.drop(['Maximo', 'Minimo', 'Dia', 'Abreviatura', 'ComponenteId', 'LocalMedicaoId', 'Nome', 'SistemaId'], axis=1, errors='ignore', inplace=True)
        temp_df.columns = [col if col == 'Data' else f"{j}_{col}" for col in temp_df.columns]
        temp_df['Data'] = pd.to_datetime(temp_df['Data'])
        df_full = safe_merge(df_full, temp_df)
        
    if not df_full.empty:
        df_full.set_index('Data', inplace=True)
    return df_full

# ==============================================================================
# 4. EXECUÇÃO DO LOOP PRINCIPAL (MÊS A MÊS)
# ==============================================================================
datas_loop = pd.date_range(start=DATA_INICIO, end=DATA_FIM, freq='D')
dfs_list = []
print(datas_loop)

print(f"\nIniciando a busca de dados de {DATA_INICIO.strftime('%Y-%m-%d')} até {DATA_FIM.strftime('%Y-%m-%d')}...")

for start_of_month in datas_loop:
    _, last_day_of_month = calendar.monthrange(start_of_month.year, start_of_month.month)
    end_of_month = date(start_of_month.year, start_of_month.month, last_day_of_month)
    
    # Garante que a data final do chunk não ultrapasse a DATA_FIM geral
    if end_of_month > DATA_FIM:
        end_of_month = DATA_FIM

    print(f"\nBuscando dados para o período: {start_of_month.strftime('%Y-%m-%d')} a {end_of_month.strftime('%Y-%m-%d')}")
    
    url = f"http://mananciais.sabesp.com.br/api/Mananciais/RepresasSistemasNivel/{start_of_month.strftime('%Y-%m-%d')}/{end_of_month.strftime('%Y-%m-%d')}/{SISTEMA_ID}"
    jsn = get_json(url)

    if jsn is None:
        print("  !!! Falha ao obter JSON. Pulando para o próximo mês.")
        continue

    df_base = json2df(jsn)

    # Extrai todas as tabelas de dados
    df_volumes = list_volumes(df_base)
    df_vazao = list_vazao(df_base)
    df_SE = list_SE(df_base)
    df_SC = list_SC(df_base)
    df_vazaoestruturas = list_vazaoestruturas(df_base)

    # Junta todas as tabelas do mês em uma só
    df_mes = pd.concat([df_volumes, df_vazao, df_SE, df_SC, df_vazaoestruturas], axis=1)
    dfs_list.append(df_mes)

    time.sleep(3) # Pausa para não sobrecarregar o servidor

# ==============================================================================
# 5. CONSOLIDAÇÃO FINAL E EXPORTAÇÃO
# ==============================================================================
if dfs_list:
    
    print("\nConsolidando todos os dados baixados...")
    df_final = pd.concat(dfs_list)
    
    # Remove linhas duplicadas (de sobreposição de meses, se houver) e ordena
    df_final = df_final[~df_final.index.duplicated(keep='last')]
    df_final.sort_index(inplace=True)

    # Garante que o DataFrame final cubra todo o período solicitado
    date_index = pd.date_range(start=DATA_INICIO, end=DATA_FIM, freq='D')
    print(date_index)
    df_final = df_final.reindex(date_index)

    # Exportação para CSV
    data_path = 'data'
    os.makedirs(data_path, exist_ok=True)
    filename = f"tab_Cantareira_append.csv"
    filepath = os.path.join(data_path, filename)
    
    df_final.to_csv(
        filepath,
        index=True,
        index_label='Data',
        header=True,
        sep=';',
        decimal=',',
        date_format='%d/%m/%Y',
        encoding='utf-8-sig'
    )
    
    print(f"\n✅ BRABA! Processo finalizado com sucesso!")
    print(f"Arquivo salvo em: {filepath}")
    print("\nVisualização das 5 primeiras linhas do resultado:")

else:
    print("\n❌ Nenhuma informação foi baixada. Verifique os parâmetros e a conexão.")
    df_final = pd.DataFrame()  # Garante que df_final exista para concatenação

df_1 = pd.read_csv('./tab_cantareira.csv', sep=';', parse_dates=['Data'], dayfirst=True)


Bibliotecas importadas e configuração concluída.
DatetimeIndex(['2025-09-17', '2025-09-18'], dtype='datetime64[ns]', freq='D')

Iniciando a busca de dados de 2025-09-17 até 2025-09-18...

Buscando dados para o período: 2025-09-17 a 2025-09-18
  Fazendo requisição para: http://mananciais.sabesp.com.br/api/Mananciais/RepresasSistemasNivel/2025-09-17/2025-09-18/0


/tmp/ipykernel_30163/1220124511.py:65: FutureWarning: Passing literal json to 'read_json' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_json(jsn)



Buscando dados para o período: 2025-09-18 a 2025-09-18
  Fazendo requisição para: http://mananciais.sabesp.com.br/api/Mananciais/RepresasSistemasNivel/2025-09-18/2025-09-18/0


/tmp/ipykernel_30163/1220124511.py:65: FutureWarning: Passing literal json to 'read_json' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_json(jsn)



Consolidando todos os dados baixados...
DatetimeIndex(['2025-09-17', '2025-09-18'], dtype='datetime64[ns]', freq='D')

✅ BRABA! Processo finalizado com sucesso!
Arquivo salvo em: data/tab_Cantareira_append.csv

Visualização das 5 primeiras linhas do resultado:
            Atibainha_Nivel  Atibainha_Volume  Atibainha_Chuva  \
2025-09-17           782.96        219.082735              0.0   
2025-09-18           782.97        219.271067              0.0   

            Atibainha_ChuvaAcumuladaMensal  Atibainha_QJusante  \
2025-09-17                             0.0                 4.5   
2025-09-18                             0.0                 4.2   

            Atibainha_VolumeMaximo  Atibainha_VolumeMinimo  \
2025-09-17              295.456327              199.204147   
2025-09-18              295.456327              199.204147   

            Atibainha_VolumePorcentagem  Atibainha_VolumeOperacional  \
2025-09-17                    20.652610                    19.878588   
2025-09-1

In [32]:
df = pd.concat([df_1, df_final], axis=0, ignore_index=True)
display(df)

,Data,Atibainha_Nivel,Atibainha_Volume,Atibainha_Chuva,Atibainha_ChuvaAcumuladaMensal,Atibainha_QJusante,Atibainha_VolumeMaximo,Atibainha_VolumeMinimo,Atibainha_VolumePorcentagem,Atibainha_VolumeOperacional,...,QT6_Unidade,QT6_Valor,QT7_Unidade,QT7_Valor,F_25bT_Unidade,F_25bT_Valor,QPS_SC_Unidade,QPS_SC_Valor,QSC_PS_Unidade,QSC_PS_Valor
0,2000-01-01,"784,0","241,46810900320997","46,8","46,8","1,08","301,5297678229108","201,36797647183994","40,035358783488164","40,100132531370036",...,m3/s,"27,7",m3/s,"29,0",NaN,NaN,NaN,NaN,NaN,NaN
1,2000-01-02,"784,11","243,65115119508118","23,6","70,4","1,08","301,5297678229108","201,36797647183994","42,21487470709976","42,283174723241245",...,m3/s,"27,7",m3/s,"29,2",NaN,NaN,NaN,NaN,NaN,NaN
2,2000-01-03,"784,25","246,444218280958","49,6","120,0","1,09","301,5297678229108","201,36797647183994","45,003430151447816","45,07624180911807",...,m3/s,"8,9",m3/s,"13,2",NaN,NaN,NaN,NaN,NaN,NaN
3,2000-01-04,"784,31","247,64627171622124","16,6","136,6","1,09","301,5297678229108","201,36797647183994","46,20354190968303","46,2782952443813",...,m3/s,"2,8",m3/s,"5,2",NaN,NaN,NaN,NaN,NaN,NaN
4,2000-01-05,"784,36","248,65028659944073","29,1","165,7","1,09","301,5297678229108","201,36797647183994","47,20593500756642","47,282310127600795",...,m3/s,"1,6",m3/s,"3,7",NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9388,2025-09-14,"783,03","220,4027042284223","0,0","0,0","4,5","295,4563270203307","199,2041470748436","22,02397614847227","21,19855715357869",...,m3/s,"21,49",m3/s,"28,5",m3/s,"3,906",m3/s,"7,558",m3/s,"0,0"
9389,2025-09-15,"783,0","219,83653371520987","0,0","0,0","4,5","295,4563270203307","199,2041470748436","21,435760366208346","20,632386640366263",...,m3/s,"22,3",m3/s,"29,1",m3/s,"3,895",m3/s,"7,553",m3/s,"0,0"
9390,2025-09-16,"782,97","219,27106707447348","0,0","0,0","4,5","295,4563270203307","199,2041470748436","20,84827586346083","20,066919999629874",...,m3/s,"22,006",m3/s,"26,7",m3/s,"3,918",m3/s,"7,561",m3/s,"0,0"
9391,NaT,782.96,219.082735,0.0,0.0,4.5,295.456327,199.204147,20.65261,19.878588,...,m3/s,21.81,m3/s,26.1,m3/s,4.02,m3/s,7.55,m3/s,0.0


In [29]:
display(df_final)

,Atibainha_Nivel,Atibainha_Volume,Atibainha_Chuva,Atibainha_ChuvaAcumuladaMensal,Atibainha_QJusante,Atibainha_VolumeMaximo,Atibainha_VolumeMinimo,Atibainha_VolumePorcentagem,Atibainha_VolumeOperacional,Atibainha_VolumeTotal,...,QPS_SC_Unidade,QPS_SC_Valor,QSC_PS_Unidade,QSC_PS_Valor,QT5_Unidade,QT5_Valor,QT6_Unidade,QT6_Valor,QT7_Unidade,QT7_Valor
2000-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2000-01-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2000-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2000-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2000-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-09-14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-09-15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-09-16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-09-17,782.96,219.082735,0.0,0.0,4.5,295.456327,199.204147,20.652610,19.878588,219.082735,...,m3/s,7.55,m3/s,0.0,m3/s,27.34,m3/s,21.810,m3/s,26.10
